## EDA has been carried out simultaneously at EDA/merges.ipynb

### Import modules and data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.scraping.scrape_reservoirs import get_reservoir_province_rate_limited
from backend.scraping.coordinates_api import get_coordinates_photon
from sklearn.metrics.pairwise import haversine_distances

In [2]:
water_path = PATHS['cleaned_data_notebooks']/ 'water_cleaned.parquet'
water_non_merged_pd = pd.read_parquet(water_path)
water_non_merged_pd.head()

,date,storage,id,storage_imputed
0,1988-01-05,103,1,0
1,1988-01-12,103,1,0
2,1988-01-19,103,1,0
3,1988-01-26,103,1,0
4,1988-02-02,103,1,0


In [3]:
reservoirs_path = PATHS['cleaned_data_notebooks'] / 'reservoirs_cleaned.csv'
reservoirs_pd = pd.read_csv(reservoirs_path)
reservoirs_pd.head()

,id,scope,name,capacity,electric_flag
0,1,guadalquivir,brena,103.0,0
1,3,guadalquivir,fernandina,247.0,0
2,5,guadalquivir,puebla cazalla,87.0,0
3,6,guadalquivir,pedro marin,19.0,0
4,9,cuenca mediterranea andaluza,vinuela,170.0,0


In [4]:
detailed_reservoirs_path = PATHS['cleaned_data_notebooks'] / 'detailed_reservoirs_cleaned.csv'
detailed_reservoirs_pd = pd.read_csv(detailed_reservoirs_path)
detailed_reservoirs_pd.head()

,name,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,canelles,41.978556,0.612326,ebro,rio noguera ribagorcana,NaN,NaN,https://www.wikidata.org/wiki/Q1033116,huesca,aragon,presa fabrica boveda,508.0,151.0,https://sig.mapama.gob.es/WebServices/clientew...
1,portas,42.114438,-7.209227,mino sil,rio camba,NaN,NaN,NaN,ourense,galicia,presa fabrica boveda,886.0,141.0,https://sig.mapama.gob.es/WebServices/clientew...
2,tous,39.132013,-0.651384,jucar,rio jucar,NaN,NaN,NaN,valència valencia,comunitat valenciana,presa materiales sueltos zonificada o nucleo,162.5,135.5,https://sig.mapama.gob.es/WebServices/clientew...
3,susqueda,41.979226,2.526874,cuencas internas cataluna,riu ter,NaN,NaN,https://www.wikidata.org/wiki/Q7131953,girona,cataluna,presa fabrica boveda,357.0,135.0,https://sig.mapama.gob.es/WebServices/clientew...
4,belesar,42.628777,-7.712394,mino sil,rio mino,https://www.google.com/search?kgmid=/g/1214fx9h,NaN,https://www.wikidata.org/wiki/Q3375700,lugo,galicia,presa fabrica boveda,332.0,132.0,https://sig.mapama.gob.es/WebServices/clientew...


### Create the merged dataframe from reservoirs, with even the non-matching entries

#### Repeated correspondences:

In [5]:
repeated_in_detailed = ['agrio', 'aguilar campoo', 'algeciras  rambla', 'arcos', 'bachimana alto', 'banos montemayor', 'burguillo', 'castrelo mino', 'chandreja', 'grado i', 'guadalteba', 'ibon ip', 'jerte', 'malpasillo jauja', 'molinos matachel', 'monteagudo vicarias', 'lago negro', 'peares', 'puentes iv', 'sant ponc', 'torre abraham', 'tous', 'vilagudin', 'zahara', 'juan benet']
repeated_in_reservoirs = ['agrio (aznalcollar)', 'aguilar', 'algeciras', 'arcos frontera', 'bachimana (lago)', 'banos', 'burguillo   puente nuevo', 'castrelo', 'chandrexa', 'grado', 'guadalhorce guadalteba', 'ip', 'jerte   plasencia', 'malpasillo ( jauja )', 'molinos', 'vicarias', 'negro (lago)', 'peares  os', 'puentes', 'sant pons', 'torre abrahan', 'tous   ribera', 'villagudin', 'zahara gastor', 'porma (juan benet)']

Now a mapping is created between both names for the same reservoir

In [6]:
series_repeated = pd.Series(repeated_in_detailed, index=repeated_in_reservoirs)
series_repeated

agrio (aznalcollar)                       agrio
aguilar                          aguilar campoo
algeciras                     algeciras  rambla
arcos frontera                            arcos
bachimana (lago)                 bachimana alto
banos                          banos montemayor
burguillo   puente nuevo              burguillo
castrelo                          castrelo mino
chandrexa                             chandreja
grado                                   grado i
guadalhorce guadalteba               guadalteba
ip                                      ibon ip
jerte   plasencia                         jerte
malpasillo ( jauja )           malpasillo jauja
molinos                        molinos matachel
vicarias                    monteagudo vicarias
negro (lago)                         lago negro
peares  os                               peares
puentes                              puentes iv
sant pons                             sant ponc
torre abrahan                     torre 

Converting the names of reservoirs_pd as the names at detailed_reservoirs_pd

In [7]:
reservoirs_pd['name'] = reservoirs_pd['name'].map(series_repeated).fillna(reservoirs_pd['name'])
reservoirs_pd

,id,scope,name,capacity,electric_flag
0,1,guadalquivir,brena,103.0,0
1,3,guadalquivir,fernandina,247.0,0
2,5,guadalquivir,puebla cazalla,87.0,0
3,6,guadalquivir,pedro marin,19.0,0
4,9,cuenca mediterranea andaluza,vinuela,170.0,0
...,...,...,...,...,...
396,735,jucar,bellus,69.0,0
397,739,ebro,laverne,38.0,0
398,741,ebro,pena,19.0,0
399,743,ebro,santa maria belsue,14.0,0


We remain with the dataframe reservoirs as the left one, because it has the ID so that it can be paired with water.csv

In [8]:
final_merged_df = pd.merge(reservoirs_pd, detailed_reservoirs_pd, how='left', on='name')
final_merged_df.head(10)

,id,scope,name,capacity,electric_flag,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,1,guadalquivir,brena,103.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,guadalquivir,fernandina,247.0,0,38.179646,-3.570224,guadalquivir,rio guarrizas,NaN,NaN,NaN,jaen,andalucia,presa fabrica gravedad (hormigon vibrado),719.55,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,5,guadalquivir,puebla cazalla,87.0,0,37.129772,-5.243309,guadalquivir,rio corbones,NaN,NaN,NaN,sevilla,andalucia,presa fabrica gravedad (hormigon compactado),218.25,NaN,https://sig.mapama.gob.es/WebServices/clientew...
3,6,guadalquivir,pedro marin,19.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9,cuenca mediterranea andaluza,vinuela,170.0,0,36.860400,-4.164435,cuencas mediterraneas andaluzas,rio guaro,https://www.google.com/search?kgmid=/g/120jr2sh,NaN,https://www.wikidata.org/wiki/Q5830515,malaga,andalucia,presa materiales sueltos pantalla hormigon,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
5,11,cuenca mediterranea andaluza,rules,111.0,0,36.860510,-3.495430,cuencas mediterraneas andaluzas,rio guadalfeo o rio cadiar,https://www.google.com/search?kgmid=/g/121vx8sj,NaN,https://www.wikidata.org/wiki/Q5369455,granada,andalucia,presa fabrica arco gravedad,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
6,12,guadiana,burdalo,79.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,14,guadiana,cabezuela,43.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,18,mino sil,castrelo mino,60.0,1,42.292500,-8.116410,mino sil,rio mino,NaN,NaN,NaN,ourense,galicia,presa fabrica gravedad (hormigon vibrado),90.00,30.00,https://sig.mapama.gob.es/WebServices/clientew...
9,19,mino sil,cenza,40.0,1,42.195847,-7.246825,mino sil,rio conselo ou da cenza,NaN,NaN,NaN,ourense,galicia,presa fabrica gravedad (hormigon compactado),1343.50,49.23,https://sig.mapama.gob.es/WebServices/clientew...


In [9]:
print(f"The paired reservoirs of the merged dataframe is {len(final_merged_df[final_merged_df['longitude'].notna()])}")

The paired reservoirs of the merged dataframe is 297


### Imputing missing values with Scraping

#### Firstly, a function is going to be created so that, if the province of one reservoir has already been scraped, it can be reused.


In [10]:
saved_df_path = PATHS['pre_EDA'] / 'merges_for_EDA.csv'
saved_df = pd.read_csv(saved_df_path)

In [11]:
def get_reservoir_province_reusing_data(reservoir_name):
    if saved_df.loc[saved_df['name'] == reservoir_name, 'province'].notna().any():
        return saved_df.loc[saved_df['name'] == reservoir_name, 'province'].values[0]
    return get_reservoir_province_rate_limited(reservoir_name)

In [12]:
print(f"There are {final_merged_df['province'].isna().sum()} missing values in the 'province' column.")

There are 104 missing values in the 'province' column.


Imputing missing values using the scraping function

In [13]:
mask = final_merged_df['province'].isna()
final_merged_df['province'] = final_merged_df['province'].where(final_merged_df['province'].notna(), final_merged_df['name'].apply(get_reservoir_province_reusing_data))

# DO NOT TOUCH

#### Importing the None ones

In [23]:
final_merged_df[final_merged_df['province'].isnull()]

,id,scope,name,capacity,electric_flag,latitude,longitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
159,312,ebro,certescans,16,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
282,536,tajo,vellon (pedrezuela),41,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
362,669,guadiana,cornalbo,11,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
376,688,ebro,lechago,18,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN


Looking one by one where they are located:

In [24]:
mask_None = final_merged_df['province'].isnull()
name_None = ['certescans', 'vellon (pedrezuela)', 'cornalbo', 'lechago']
province_None = ['lleida', 'madrid', 'badajoz', 'teruel']
series_None = pd.Series(province_None, index=name_None)
final_merged_df['province'] = final_merged_df['province'].where(final_merged_df['province'].notna(), final_merged_df['name'].map(series_None))
final_merged_df[mask_None]

,id,scope,name,capacity,electric_flag,latitude,longitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
159,312,ebro,certescans,16,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lleida,NaN,NaN,NaN,NaN,NaN
282,536,tajo,vellon (pedrezuela),41,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,madrid,NaN,NaN,NaN,NaN,NaN
362,669,guadiana,cornalbo,11,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,badajoz,NaN,NaN,NaN,NaN,NaN
376,688,ebro,lechago,18,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,teruel,NaN,NaN,NaN,NaN,NaN


# UNTIL HERE

### Unify the province names

First sight:

In [15]:
final_merged_df['province'].sort_values().unique()

array(['a coruna', 'alacant alicante', 'alava', 'albacete', 'almeria',
       'araba alava', 'asturias', 'avila', 'badajoz', 'barcelona',
       'burgos', 'caceres', 'cadiz', 'cantabria', 'castello castellon',
       'ciudad real', 'cordoba', 'cuenca', 'gipuzkoa guipuzcoa', 'girona',
       'granada', 'guadalajara', 'guipuzcoa', 'huelva', 'huesca', 'jaen',
       'la coruna', 'la rioja', 'las palmas', 'leon', 'lleida', 'lugo',
       'madrid', 'malaga', 'murcia', 'navarra', 'orense', 'ourense',
       'palencia', 'pontevedra', 'rioja', 'salamanca', 'segovia',
       'sevilla', 'soria', 'tarragona', 'teruel', 'toledo', 'valencia',
       'valladolid', 'valència valencia', 'zamora', 'zaragoza'],
      dtype=object)

In [16]:
unified_provinces = {
    'alacant alicante': 'alicante',
    'araba alava': 'alava',
    'castello castellon': 'castellon',
    'gipuzkoa guipuzcoa': 'guipuzcoa',
    'ourense': 'orense',
    'valència valencia': 'valencia',
    'a coruna': 'la coruna',
    'rioja': 'la rioja'
}

final_merged_df['province'] = final_merged_df['province'].map(unified_provinces).fillna(final_merged_df['province'])
final_merged_df['province'].sort_values().unique()

array(['alava', 'albacete', 'alicante', 'almeria', 'asturias', 'avila',
       'badajoz', 'barcelona', 'burgos', 'caceres', 'cadiz', 'cantabria',
       'castellon', 'ciudad real', 'cordoba', 'cuenca', 'girona',
       'granada', 'guadalajara', 'guipuzcoa', 'huelva', 'huesca', 'jaen',
       'la coruna', 'la rioja', 'las palmas', 'leon', 'lleida', 'lugo',
       'madrid', 'malaga', 'murcia', 'navarra', 'orense', 'palencia',
       'pontevedra', 'salamanca', 'segovia', 'sevilla', 'soria',
       'tarragona', 'teruel', 'toledo', 'valencia', 'valladolid',
       'zamora', 'zaragoza'], dtype=object)

In [17]:
province_to_community = {
    'alava': 'pais vasco',
    'albacete': 'castilla   mancha',
    'alicante': 'comunitat valenciana',
    'almeria': 'andalucia',
    'asturias': 'principado asturias',
    'avila': 'castilla y leon',
    'badajoz': 'extremadura',
    'barcelona': 'cataluna',
    'burgos': 'castilla y leon',
    'caceres': 'extremadura',
    'cadiz': 'andalucia',
    'cantabria': 'cantabria',
    'castellon': 'comunitat valenciana',
    'ciudad real': 'castilla   mancha',
    'cordoba': 'andalucia',
    'cuenca': 'castilla   mancha',
    'girona': 'cataluna',
    'granada': 'andalucia',
    'guadalajara': 'castilla   mancha',
    'guipuzcoa': 'pais vasco',
    'huelva': 'andalucia',
    'huesca': 'aragon',
    'jaen': 'andalucia',
    'la coruna': 'galicia',
    'la rioja': 'rioja',
    'las palmas': 'canarias',
    'leon': 'castilla y leon',
    'lleida': 'cataluna',
    'lugo': 'galicia',
    'madrid': 'comunidad madrid',
    'malaga': 'andalucia',
    'murcia': 'region murcia',
    'navarra': 'comunidad foral navarra',
    'orense': 'galicia',
    'palencia': 'castilla y leon',
    'pontevedra': 'galicia',
    'salamanca': 'castilla y leon',
    'santa cruz de tenerife': 'canarias',
    'segovia': 'castilla y leon',
    'sevilla': 'andalucia',
    'soria': 'castilla y leon',
    'tarragona': 'cataluna',
    'teruel': 'aragon',
    'toledo': 'castilla   mancha',
    'valencia': 'comunitat valenciana',
    'valladolid': 'castilla y leon',
    'vizcaya': 'pais vasco',
    'zamora': 'castilla y leon',
    'ceuta': 'ceuta',
    'melilla': 'melilla',
    'balears': 'islas baleares',
    'zaragoza': 'aragon'
}

final_merged_df['autonomous_community'] = final_merged_df['province'].map(province_to_community)

### Saving dataframe for EDA

In [23]:
merges_for_EDA_path = PATHS['pre_EDA'] / 'merges_for_EDA.csv'
merges_for_EDA_path.parent.mkdir(parents=True, exist_ok=True)
final_merged_df.to_csv(merges_for_EDA_path, index=False)

### During EDA workflow

Merge of water with reservoirs:

In [19]:
water_pd = pd.merge(water_non_merged_pd, final_merged_df[['id', 'capacity', 'crest_elevation', 'province', 'autonomous_community']], on='id', how='left')
water_pd.head()

,date,storage,id,storage_imputed,capacity,crest_elevation,province,autonomous_community
0,1988-01-05,103,1,0,103.0,NaN,cordoba,andalucia
1,1988-01-12,103,1,0,103.0,NaN,cordoba,andalucia
2,1988-01-19,103,1,0,103.0,NaN,cordoba,andalucia
3,1988-01-26,103,1,0,103.0,NaN,cordoba,andalucia
4,1988-02-02,103,1,0,103.0,NaN,cordoba,andalucia


### Fixing inconsistencies

- With storage higher than capacity:

In [20]:
mask = water_pd['storage'] > water_pd['capacity']
water_pd.loc[mask, 'storage'] = water_pd.loc[mask, 'capacity']
water_non_merged_pd.loc[mask, 'storage'] = water_pd.loc[mask, 'capacity']

The inconsistencies have also been fixed in the water_non_merged_pd dataframe

- Deleting reservoirs that don't have data up to date: (they turn out to be redundant, there are 374 large reservoirs in Spain as August 2025)

In [21]:
max_date = water_pd['date'].max()
print(f"The maximum date in the water dataframe is {max_date}.")

The maximum date in the water dataframe is 2024-09-24 00:00:00.


In [22]:
list_final_reservoirs = water_pd[water_pd['date'] == max_date]['id'].values

Updating all the dataframes to ensure consistency:

In [23]:
water_pd = water_pd[water_pd['id'].isin(list_final_reservoirs)]
water_non_merged_pd = water_non_merged_pd[water_non_merged_pd['id'].isin(list_final_reservoirs)]
reservoirs_pd = reservoirs_pd[reservoirs_pd['id'].isin(list_final_reservoirs)]
final_merged_df = final_merged_df[final_merged_df['id'].isin(list_final_reservoirs)]

### Imputing missing values

In [25]:
final_merged_df.isna().sum()

id                        0
scope                     0
name                      0
capacity                  0
electric_flag             0
longitude                86
latitude                 86
basin                    86
riverbed                 86
google                  277
openstreetmap           345
wikidata                219
province                  0
autonomous_community      0
type                     86
crest_elevation          86
dam_height              235
report                   86
dtype: int64

### Impute longitude and latitude with a geographical API

In [26]:
reservoirs_without_coordinates = final_merged_df[final_merged_df['latitude'].isna()]
reservoirs_without_coordinates

,id,scope,name,capacity,electric_flag,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
6,12,guadiana,burdalo,79.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,caceres,extremadura,NaN,NaN,NaN,NaN
7,14,guadiana,cabezuela,43.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ciudad real,castilla mancha,NaN,NaN,NaN,NaN
24,50,ebro,puente santolea,18.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,teruel,aragon,NaN,NaN,NaN,NaN
25,51,ebro,san salvador,137.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,huesca,aragon,NaN,NaN,NaN,NaN
27,54,ebro,sistema lagos espot,10.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lleida,cataluna,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,688,ebro,lechago,18.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,teruel,aragon,NaN,NaN,NaN,NaN
382,714,tinto. odiel y piedras,odiel,8.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,huelva,andalucia,NaN,NaN,NaN,NaN
383,715,mino sil,bao,238.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,orense,galicia,NaN,NaN,NaN,NaN
384,716,mino sil,san estevo,213.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,caceres,extremadura,NaN,NaN,NaN,NaN


#### Define a function so that the API is only used the first time:

In [27]:
saved_df_path = PATHS['definitive_notebooks'] / 'reservoirs_merged.parquet'
saved_df = pd.read_parquet(saved_df_path)
saved_df

,id,scope,name,capacity,electric_flag,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,3,guadalquivir,fernandina,247.0,0,38.179646,-3.570224,guadalquivir,rio guarrizas,None,None,None,jaen,andalucia,presa fabrica gravedad (hormigon vibrado),719.55,NaN,https://sig.mapama.gob.es/WebServices/clientew...
1,5,guadalquivir,puebla cazalla,87.0,0,37.129772,-5.243309,guadalquivir,rio corbones,None,None,None,sevilla,andalucia,presa fabrica gravedad (hormigon compactado),218.25,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,9,cuenca mediterranea andaluza,vinuela,170.0,0,36.860400,-4.164435,cuencas mediterraneas andaluzas,rio guaro,https://www.google.com/search?kgmid=/g/120jr2sh,None,https://www.wikidata.org/wiki/Q5830515,malaga,andalucia,presa materiales sueltos pantalla hormigon,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
3,11,cuenca mediterranea andaluza,rules,111.0,0,36.860510,-3.495430,cuencas mediterraneas andaluzas,rio guadalfeo o rio cadiar,https://www.google.com/search?kgmid=/g/121vx8sj,None,https://www.wikidata.org/wiki/Q5369455,granada,andalucia,presa fabrica arco gravedad,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
4,12,guadiana,burdalo,79.0,0,-1.112043,42.606125,jucar,riu amadorio,None,None,None,caceres,extremadura,None,320.65,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,735,jucar,bellus,69.0,0,38.939709,-0.477465,jucar,rio albaida,https://www.google.com/search?kgmid=/g/121rjw4w,None,https://www.wikidata.org/wiki/Q5670515,valencia,comunitat valenciana,presa fabrica gravedad (hormigon vibrado),159.00,43.0,https://sig.mapama.gob.es/WebServices/clientew...
370,739,ebro,laverne,38.0,0,42.088538,-1.090922,ebro,vertiente por margen derecha barranco calcina,None,None,None,zaragoza,aragon,presa materiales sueltos zonificada o nucleo,403.00,17.5,https://sig.mapama.gob.es/WebServices/clientew...
371,741,ebro,pena,19.0,0,42.383383,-0.735550,ebro,rio gallego,None,None,None,huesca,aragon,presa fabrica mamposteria,541.00,61.0,https://sig.mapama.gob.es/WebServices/clientew...
372,743,ebro,santa maria belsue,14.0,0,42.302380,-0.347296,ebro,rio flumen,None,None,None,huesca,aragon,presa fabrica mamposteria,896.40,52.0,https://sig.mapama.gob.es/WebServices/clientew...


In [28]:
def get_coordinates_reusing_data(reservoir_list):
    if saved_df['latitude'].isna().any():
        return get_coordinates_photon(reservoir_list)
    else:
        path = PATHS['raw_data_notebooks'] / 'coordinates.csv'
        coordinates = pd.read_csv(path)
        return coordinates

In [29]:
coordinates = get_coordinates_reusing_data(reservoirs_without_coordinates['name'].values)
coordinates

,name,latitude,longitude
0,burdalo,42.606125,-1.112043
1,cabezuela,38.711212,-3.262285
2,puente santolea,40.742248,-0.353889
3,san salvador,41.781409,0.201096
4,sistema lagos espot,NaN,NaN
...,...,...,...
81,lechago,40.956354,-1.286902
82,odiel,37.792175,-6.618268
83,bao,41.339855,0.093840
84,san estevo,42.079895,0.764046


Saving coordinates so that it's only calculated once

In [ ]:
raw_coordinates_path = PATHS['raw_data_notebooks'] / 'coordinates.csv'
coordinates.to_csv(raw_coordinates_path, index=False)

In [31]:
coordinates_df = coordinates.copy()
coordinates_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       86 non-null     object 
 1   latitude   81 non-null     float64
 2   longitude  81 non-null     float64
dtypes: float64(2), object(1)
memory usage: 2.1+ KB


Seeing which reservoirs weren't completed with the API

In [32]:
coordinates_df[coordinates_df['latitude'].isna()]

,name,latitude,longitude
4,sistema lagos espot,NaN,NaN
33,agavanzal nª sª,NaN,NaN
37,sistema aguas limpias,NaN,NaN
59,sistema alto caldares,NaN,NaN
77,tremp o talarn,NaN,NaN


The second reservoir has an issue with it name, we will fix it in the next notebook

In [33]:
rows = [
        {'name': 'sistema lagos espot', 'latitude': 42.581771, 'longitude': 1.003018},
        {'name': 'agavanzal', 'latitude': 41.979600, 'longitude': -6.200336},
        {'name': 'sistema aguas limpias', 'latitude': 42.793868, 'longitude': -0.329262},
        {'name': 'sistema alto caldares', 'latitude': 42.427684, 'longitude': -0.227609},
        {'name': 'tremp o talarn', 'latitude': 42.204670, 'longitude': 0.949327}
        ]

Filling the missing coordinates by hand

In [34]:
coordinates_df = pd.concat([pd.DataFrame(rows), coordinates_df], ignore_index=True)
coordinates_df = coordinates_df[coordinates_df['latitude'].notna()]
coordinates_df.set_index('name', inplace=True)
coordinates_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 86 entries, sistema lagos espot to portaje
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   latitude   86 non-null     float64
 1   longitude  86 non-null     float64
dtypes: float64(2)
memory usage: 2.0+ KB


Fixing the second reservoir name

In [35]:
final_merged_df.loc[final_merged_df['id'] == 342, 'name'] = 'agavanzal'

Imputing all the coordinates of the reservoirs

In [37]:
name_to_latitude = coordinates_df['latitude'].to_dict()
name_to_longitude = coordinates_df['longitude'].to_dict()

mask = final_merged_df['name'].isin(coordinates_df.index)
final_merged_df.loc[mask, 'latitude'] = final_merged_df.loc[mask, 'name'].map(name_to_latitude)
final_merged_df.loc[mask, 'longitude'] = final_merged_df.loc[mask, 'name'].map(name_to_longitude)

In [39]:
final_merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 374 entries, 1 to 400
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    374 non-null    int64  
 1   scope                 374 non-null    object 
 2   name                  374 non-null    object 
 3   capacity              374 non-null    float64
 4   electric_flag         374 non-null    int64  
 5   longitude             374 non-null    float64
 6   latitude              374 non-null    float64
 7   basin                 288 non-null    object 
 8   riverbed              288 non-null    object 
 9   google                97 non-null     object 
 10  openstreetmap         29 non-null     object 
 11  wikidata              155 non-null    object 
 12  province              374 non-null    object 
 13  autonomous_community  374 non-null    object 
 14  type                  288 non-null    object 
 15  crest_elevation       288 no

### Impute Crest Elevation, Riverbed and Basin with the closest non null value

In [40]:
def impute_nearest_neighbour(detailed_reservoirs_data, column_name):
    nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].isna()]
    non_nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].notna()]

    nan_coord_rads = np.radians(nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)
    non_nan_coord_rads = np.radians(non_nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)

    distances = haversine_distances(nan_coord_rads, non_nan_coord_rads) * 6371.0 
    closest_indices = distances.argmin(axis=1)

    # Create a mapping from indices to reservoir values
    reservoir_mapping = non_nan_reservoirs[column_name].iloc[closest_indices]
    detailed_reservoirs_data.loc[nan_reservoirs.index, column_name] = reservoir_mapping.values
    return detailed_reservoirs_data

In [41]:
final_merged_df = impute_nearest_neighbour(final_merged_df, 'crest_elevation')
final_merged_df = impute_nearest_neighbour(final_merged_df, 'riverbed')
final_merged_df = impute_nearest_neighbour(final_merged_df, 'basin')

In [42]:
final_merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 374 entries, 1 to 400
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    374 non-null    int64  
 1   scope                 374 non-null    object 
 2   name                  374 non-null    object 
 3   capacity              374 non-null    float64
 4   electric_flag         374 non-null    int64  
 5   longitude             374 non-null    float64
 6   latitude              374 non-null    float64
 7   basin                 374 non-null    object 
 8   riverbed              374 non-null    object 
 9   google                97 non-null     object 
 10  openstreetmap         29 non-null     object 
 11  wikidata              155 non-null    object 
 12  province              374 non-null    object 
 13  autonomous_community  374 non-null    object 
 14  type                  288 non-null    object 
 15  crest_elevation       374 no

### Save all the dataframes modified here

In [44]:
water_definitive_path = PATHS['definitive_notebooks'] / 'water_definitive.parquet'
water_non_merged_pd.to_parquet(water_definitive_path, index=False)

In [45]:
merged_reservoirs_path = PATHS['definitive_notebooks'] / 'reservoirs_merged.parquet'
final_merged_df.to_parquet(merged_reservoirs_path, index=False)

In [46]:
reservoirs_definitive_path = PATHS['definitive_notebooks'] / 'reservoirs_definitive.parquet'
reservoirs_pd.to_parquet(reservoirs_definitive_path, index=False)

In [47]:
detailed_reservoirs_definitive_path = PATHS['definitive_notebooks'] / 'detailed_reservoirs_definitive.parquet'
detailed_reservoirs_pd.to_parquet(detailed_reservoirs_definitive_path, index=False)